In [ ]:
# Solutions live in solutions/. Run from the repository root so imports and paths work.
import os, pathlib, sys
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "workshop").is_dir())
os.chdir(_root)
sys.path.insert(0, str(_root))

# Station B: drawdata

**Good for:** drawing a dataset by hand to build intuition for machine
learning, or for teaching. Works offline. By Vincent Warmerdam.

Docs: https://github.com/koaning/drawdata

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from drawdata import ScatterWidget
from sklearn.neighbors import KNeighborsClassifier

canvas = ScatterWidget(n_classes=2)
canvas

Draw some points of two classes (pick the colour on the left), then read them in Python:

In [ ]:
drawn = canvas.data_as_pandas
drawn.head()

### Your task: retrain live

Train a k-NN classifier on what you draw, and **redraw its decision boundary
every time the drawing changes**. The drawing is the synced trait `data`.

In [ ]:
neighbours = widgets.IntSlider(value=5, min=1, max=30, description="neighbours")
plot = widgets.Output()


def retrain(change=None):
    df = canvas.data_as_pandas
    if len(df) < 5 or df["label"].nunique() < 2:
        return
    model = KNeighborsClassifier(n_neighbors=min(neighbours.value, len(df))).fit(df[["x", "y"]], df["label"])
    xs = np.linspace(df.x.min() - 20, df.x.max() + 20, 150)
    ys = np.linspace(df.y.min() - 20, df.y.max() + 20, 150)
    gx, gy = np.meshgrid(xs, ys)
    grid = np.c_[gx.ravel(), gy.ravel()]
    zz = (model.predict(grid) == model.classes_[0]).reshape(gx.shape)
    with plot:
        plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.contourf(gx, gy, zz, alpha=0.3, cmap="coolwarm")
        ax.scatter(df.x, df.y, c=df["color"], s=12)
        ax.invert_yaxis()  # drawdata's y axis points down
        ax.set_title(f"k-NN, k={model.n_neighbors}, {len(df)} points")
        plt.show()


canvas.observe(retrain, names=["data"])
neighbours.observe(retrain, names="value")
retrain()
widgets.VBox([neighbours, plot])